<a href="https://colab.research.google.com/github/Shivam-vachhani/PyTorch-Projects/blob/main/ANN_On_Telecom_Churn_Dataset_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [111]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE


In [112]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [113]:
# feature engeineering
df = df.drop(columns=["customerID"])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],errors='coerce')
df.dropna(subset=["TotalCharges"],inplace=True)



In [114]:
# Encode yes and no values to 0 and 1 and if three values then 0,1,2 accordingly

Encodable_feilds_two_values=['Partner', 'Dependents', 'PhoneService',  'PaperlessBilling', 'Churn']
Encodable_feilds_three_values= ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
def encoding_values(df):
    for feature in df.columns:
        if feature in Encodable_feilds_two_values:
           df[feature] = df[feature].apply(lambda x:1 if x == "Yes" else 0)
        if feature in Encodable_feilds_three_values:
           df[feature] = df[feature].apply(lambda x:0 if x == "No phone service" or x == "No internet service" else(1 if x=="No" else 2))
    return df
df = encoding_values(df)


# Convert some categorycal features into numeric so model can understand and drop old object fields

df['is_male'] = df["gender"].apply(lambda x:1 if x=='Male' else 0)
df = pd.get_dummies(df,columns=['PaymentMethod','InternetService'],drop_first=True,dtype=int)
df['Contract'] = df["Contract"].apply(lambda x:0 if x == 'Month-to-month' else(1 if x == 'One year'else 2 ))
df = df.drop(columns=['TotalCharges','gender'])

In [115]:
df.head()

,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,MonthlyCharges,Churn,is_male,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No
0,0,1,0,1,0,0,1,2,1,1,1,1,0,1,29.85,0,0,0,1,0,0,0
1,0,0,0,34,1,1,2,1,2,1,1,1,1,0,56.95,0,1,0,0,1,0,0
2,0,0,0,2,1,1,2,2,1,1,1,1,0,1,53.85,1,1,0,0,1,0,0
3,0,0,0,45,0,0,2,1,2,2,1,1,1,0,42.30,0,1,0,0,0,0,0
4,0,0,0,2,1,1,1,1,1,1,1,1,0,1,70.70,1,0,0,1,0,1,0


In [116]:
# split data then scaled and performe SMOTE on train data
X = df.drop(columns=("Churn"))
y = df["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Smote = SMOTE(random_state=42)
X_train_res,y_train_res = Smote.fit_resample(X_train_scaled,y_train)

In [117]:
class MyCustomDataset(Dataset):

  def __init__(self,feature,label):
    self.feature = torch.tensor(feature,dtype=torch.float32)
    self.label  = torch.tensor(label,dtype=torch.long)

  def __len__(self):
    return len(self.feature)

  def __getitem__(self,index):
    return self.feature[index],self.label[index]

In [118]:
train_dataset  = MyCustomDataset(X_train_res,y_train_res.values)
test_dataset = MyCustomDataset(X_test_scaled,y_test.values)

In [119]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [134]:
class MyNN(nn.Module):

  def __init__(self,feature_size):
    super().__init__()
    self.layers = nn.Sequential(
        nn.Linear(feature_size,32),
        nn.BatchNorm1d(32),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(32,16),
        nn.BatchNorm1d(16),
        nn.ReLU(),
        nn.Dropout(0.1),
        nn.Linear(16,1)
      )

  def forward(self,x):
    x = self.layers(x)
    return x

In [135]:
lr = 0.001
epochs = 100

In [136]:
model  = MyNN(X_train_res.shape[1])

optimizer = optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)

crierion = nn.BCEWithLogitsLoss()

In [137]:
for i in range(epochs):
  total_epoch_loss = 0
  for batch_featrue,batch_label in train_loader:

    output = model(batch_featrue)

    loss = crierion(output,batch_label.unsqueeze(1).float())

    optimizer.zero_grad()
    loss.backward()

    optimizer.step()

    total_epoch_loss +=loss.item()
  avg_loss = total_epoch_loss / len(train_loader)
  print(f'Epoch: {i + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 0.5553031833015354
Epoch: 2 , Loss: 0.4840419787237543
Epoch: 3 , Loss: 0.48146580441578013
Epoch: 4 , Loss: 0.4738431246124179
Epoch: 5 , Loss: 0.46880890032039185
Epoch: 6 , Loss: 0.46777615273321
Epoch: 7 , Loss: 0.4674258067571058
Epoch: 8 , Loss: 0.46204335786200856
Epoch: 9 , Loss: 0.46302422593459197
Epoch: 10 , Loss: 0.45460659639485557
Epoch: 11 , Loss: 0.45356773973431824
Epoch: 12 , Loss: 0.45855248676303734
Epoch: 13 , Loss: 0.4525044565610444
Epoch: 14 , Loss: 0.45680763218623793
Epoch: 15 , Loss: 0.44758021474805115
Epoch: 16 , Loss: 0.44737888084415306
Epoch: 17 , Loss: 0.4520296510812399
Epoch: 18 , Loss: 0.4498388839734567
Epoch: 19 , Loss: 0.4434967154931838
Epoch: 20 , Loss: 0.44687575239932675
Epoch: 21 , Loss: 0.4456325781506461
Epoch: 22 , Loss: 0.4455214107128644
Epoch: 23 , Loss: 0.43789883930250484
Epoch: 24 , Loss: 0.4414068232056717
Epoch: 25 , Loss: 0.4437330817393815
Epoch: 26 , Loss: 0.4399695821595468
Epoch: 27 , Loss: 0.43209257894501263

In [138]:
model.eval()

MyNN(
  (layers): Sequential(
    (0): Linear(in_features=21, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.1, inplace=False)
    (8): Linear(in_features=16, out_features=1, bias=True)
  )
)

In [139]:
total = 0
correct = 0

with torch.no_grad():
  for batch_featrue,batch_label in test_loader:
    output = model(batch_featrue)
    predicted  = torch.round(torch.sigmoid(output))
    total += batch_label.size(0)
    correct += (predicted == batch_label.unsqueeze(1).float()).sum().item()
  print(correct/total)

0.7135749822316987


In [140]:
total = 0
correct = 0

with torch.no_grad():
  for batch_featrue,batch_label in train_loader:
    output = model(batch_featrue)
    predicted  = torch.round(torch.sigmoid(output))
    total += batch_label.size(0)
    correct += (predicted == batch_label.unsqueeze(1).float()).sum().item()
  print(correct/total)

0.8322033898305085
